# Point Load

load att ccf-registered points from the 8 samples 

In [ ]:
import os
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import glob
import zarr
import tifffile

from scipy.stats import rankdata
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import minimize
import scipy.sparse as sp

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

import trimesh
import open3d as o3d
import point_cloud_utils as pcu
import SimpleITK as sitk

from aind_zarr_utils.pipeline_transformed import (
    alignment_zarr_uri_and_metadata_from_zarr_or_asset_pathlike,
    mimic_pipeline_zarr_to_anatomical_stub,
    pipeline_transforms_local_paths,
)
from aind_registration_utils.ants import apply_ants_transforms_to_point_arr
import ot

from tqdm import tqdm
warnings.filterwarnings('ignore')

In [ ]:
# data naming convention
data_root = '/root/capsule/data/LC_H2B_trailmap_probabilities_and_point_calls/segmentation_and_quantification'
keywords = ['798571', '798573', '798576', '807322', '807324', '807325', '807326', '807327']

# find ccf .npy files
ccf_files = []
for kw in keywords:
    pattern = os.path.join(data_root, f'*{kw}*', '*ccf*.npy')
    ccf_files.extend([f for f in glob.glob(pattern, recursive=False)])

if len(ccf_files) == 0:
    raise FileNotFoundError(f"No matching ccf .npy files found under {data_root}")

all_points = []
for fpath in ccf_files:
    fname = os.path.basename(fpath)
    print(f"Loading file: {fpath}")
    pts = np.load(fpath).copy()
    pts *= 1000
    pts[:, 2] *= -1
    pts[:, 0] *= -1
    for row in pts:
        all_points.append({'file': fname, 'x': row[0], 'y': row[1], 'z': row[2]})

df_all_points = pd.DataFrame(all_points)

In [ ]:
df_all_points

In [ ]:
# Group by file and plot
for fname in df_all_points['file'].unique():
    df_file = df_all_points[df_all_points['file'] == fname]
    arr = df_file[['x', 'y', 'z']].values
    
    if arr.shape[0] < 10:
        print(f"Skipping {fname}: only {arr.shape[0]} points")
        continue
    
    # Randomly select 50% of points
    idx = np.random.choice(arr.shape[0], int(arr.shape[0] * 0.5), replace=False)
    arr_sub = arr[idx]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(arr_sub[:, 0], arr_sub[:, 1], alpha=0.05, s=1)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title(f'{np.max(arr_sub[:, 2]):.2f}:{fname}')
    axes[0].axis('equal')

    axes[1].scatter(arr_sub[:, 0], arr_sub[:, 2], alpha=0.05, s=1)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('z')
    axes[1].set_title(f'{np.max(arr_sub[:, 1]):.2f}: x vs z')
    axes[1].axis('equal')

    axes[2].scatter(arr_sub[:, 1], arr_sub[:, 2], alpha=0.05, s=1)
    axes[2].set_xlabel('y')
    axes[2].set_ylabel('z')
    axes[2].set_title(f'{np.max(arr_sub[:, 0]):.2f}: y vs z')
    axes[2].axis('equal')

    plt.tight_layout()
    plt.show()

# LC-specific prep and filtering

In [ ]:
grouped = df_all_points.groupby('file').size()
mean_count = grouped.mean()
std_count = grouped.std()
print(f"Mean number of cells per whole-brain: {mean_count:.2f} ± {std_count:.2f}")

In [ ]:
base_dir = '/root/capsule/data'

# Find top-level raw microscopy directories matching brain ids
top_level_matches = [
    d for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d)) and any(k in d.lower() for k in keywords)
]

print(f"Top-level data folders with zarr raw images ({len(top_level_matches)}):")
for d in top_level_matches:
    print("  ", d)  

In [ ]:
df_all_points_reflected = df_all_points.copy()

# Reflect points mask (across midline)
border = 5700
reflected_mask = df_all_points_reflected['x'] > border

# Apply reflection to x for reflected points
df_all_points_reflected.loc[reflected_mask, 'x'] = 2 * border - df_all_points_reflected.loc[reflected_mask, 'x']

# Add a new column to indicate reflection status; aka hemisphere identity
df_all_points_reflected['reflected'] = reflected_mask.astype(int)

# Create a boolean mask for filtering based on crop box
filtered_reflected_mask = (
    (df_all_points_reflected['y'] > 9000) &
    (df_all_points_reflected['y'] < 12000) &
    (df_all_points_reflected['x'] > 3000) &
    (df_all_points_reflected['z'] > 3000) &
    ~((df_all_points_reflected['z'] > 4800) & (df_all_points_reflected['y'] > 10800))
)

df_all_points_reflected['filtered_reflected'] = filtered_reflected_mask.astype(int)
LC_only_points = df_all_points_reflected[df_all_points_reflected['filtered_reflected'] == 1]

# Local density mapping

In [ ]:
# local density calculation
coords = LC_only_points[['x', 'y', 'z']].values
nbrs = NearestNeighbors(n_neighbors=101, algorithm='auto').fit(coords)
distances, _ = nbrs.kneighbors(coords)
# Exclude distance to self
LC_only_points['kNN'] = distances[:, 1:101].mean(axis=1)

# Compute percentiles for LC_only_points['kNN'] with respect to all kNN values
percentiles = rankdata(LC_only_points['kNN'], method='average') / len(LC_only_points) * 100
LC_only_points['kNN_percentile'] = percentiles

In [ ]:
LC_only_points

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=LC_only_points['y'],
        y=LC_only_points['x'],
        z=LC_only_points['z'],
        mode='markers',
        marker=dict(
            size=2,
            color=LC_only_points['kNN_percentile'],
            colorscale='Viridis',
            opacity=0.2,
            colorbar=dict(title='kNN Percentile')
        ),
        name='LC_only_points'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='y',
        yaxis_title='x',
        zaxis_title='z'
    ),
    title='LC_only_points Colorcoded by kNN Percentile'
)
fig.show()

In [ ]:
def plot_kNN_percentile(threshold):
    highlight_mask = LC_only_points['kNN_percentile'] < threshold
    n_total = len(LC_only_points)
    n_sample = min(10000, n_total)
    sample_idx = np.random.choice(n_total, n_sample, replace=False)
    sample = LC_only_points.iloc[sample_idx]
    highlight_sample = sample[highlight_mask.iloc[sample_idx]]
    non_highlight_sample = sample[~highlight_mask.iloc[sample_idx]]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(non_highlight_sample['x'], non_highlight_sample['y'], s=2, c='gray', alpha=0.05, label=f'kNN ≥ {threshold:.0f}')
    axes[0].scatter(highlight_sample['x'], highlight_sample['y'], s=2, c='red', alpha=0.1, label=f'kNN < {threshold:.0f}')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('y')
    axes[0].set_title('x vs y')
    axes[0].axis('equal')
    axes[0].legend()

    axes[1].scatter(non_highlight_sample['x'], non_highlight_sample['z'], s=2, c='gray', alpha=0.05)
    axes[1].scatter(highlight_sample['x'], highlight_sample['z'], s=2, c='red', alpha=0.1)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('z')
    axes[1].set_title('x vs z')
    axes[1].axis('equal')
    axes[1].invert_yaxis()

    axes[2].scatter(non_highlight_sample['y'], non_highlight_sample['z'], s=2, c='gray', alpha=0.05)
    axes[2].scatter(highlight_sample['y'], highlight_sample['z'], s=2, c='red', alpha=0.1)
    axes[2].set_xlabel('y')
    axes[2].set_ylabel('z')
    axes[2].set_title('y vs z')
    axes[2].axis('equal')
    axes[2].invert_yaxis()

    plt.tight_layout()
    plt.show()

slider = widgets.FloatSlider(
    value=67,
    min=0,
    max=100,
    step=1,
    description='kNN percentile threshold:',
    continuous_update=False
)
widgets.interact(plot_kNN_percentile, threshold=slider)

In [ ]:
# Sample the data once
n_total = len(LC_only_points)
n_sample = min(10000, n_total)
sample_idx = np.random.choice(n_total, n_sample, replace=False)
sample = LC_only_points.iloc[sample_idx].copy()

# Create figure with subplots
fig = make_subplots(rows=1, cols=3, subplot_titles=('x vs y', 'x vs z', 'y vs z'))

# Add traces for each threshold
thresholds = list(range(0, 101, 1))

for thresh in thresholds:
    mask = sample['kNN_percentile'] < thresh
    highlight = sample[mask]
    non_highlight = sample[~mask]
    
    visible = (thresh == 67)  # Default visible
    
    # Plot 1: x vs y
    fig.add_trace(go.Scatter(x=non_highlight['x'], y=non_highlight['y'], mode='markers',
                              marker=dict(size=3, color='gray', opacity=0.3),
                              name=f'≥{thresh}', visible=visible, showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=highlight['x'], y=highlight['y'], mode='markers',
                              marker=dict(size=3, color='red', opacity=0.5),
                              name=f'<{thresh}', visible=visible, showlegend=False), row=1, col=1)
    
    # Plot 2: x vs z
    fig.add_trace(go.Scatter(x=non_highlight['x'], y=non_highlight['z'], mode='markers',
                              marker=dict(size=3, color='gray', opacity=0.3),
                              visible=visible, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=highlight['x'], y=highlight['z'], mode='markers',
                              marker=dict(size=3, color='red', opacity=0.5),
                              visible=visible, showlegend=False), row=1, col=2)
    
    # Plot 3: y vs z
    fig.add_trace(go.Scatter(x=non_highlight['y'], y=non_highlight['z'], mode='markers',
                              marker=dict(size=3, color='gray', opacity=0.3),
                              visible=visible, showlegend=False), row=1, col=3)
    fig.add_trace(go.Scatter(x=highlight['y'], y=highlight['z'], mode='markers',
                              marker=dict(size=3, color='red', opacity=0.5),
                              visible=visible, showlegend=False), row=1, col=3)

# Create slider steps
steps = []
for i, thresh in enumerate(thresholds):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)}],
        label=str(thresh)
    )
    # Each threshold has 6 traces (2 per subplot)
    for j in range(6):
        step["args"][0]["visible"][i * 6 + j] = True
    steps.append(step)

sliders = [dict(
    active=thresholds.index(67),
    currentvalue={"prefix": "kNN percentile threshold: "},
    pad={"t": 50},
    steps=steps
)]

fig.update_layout(
    sliders=sliders,
    height=500,
    width=1200,
    title="kNN Percentile Threshold Explorer"
)

# Invert y-axis for plots 2 and 3
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=3)

# Save as standalone HTML
fig.write_html('/root/capsule/results/kNN_percentile_explorer.html', include_plotlyjs=True)
print("Saved interactive HTML to /root/capsule/results/kNN_percentile_explorer.html")

In [ ]:
# Generate mask for selected points
mask = (LC_only_points['kNN_percentile'] > 10) & (LC_only_points['kNN_percentile'] < slider.value)
selected_points = LC_only_points.loc[mask, ['x', 'y', 'z']].values
print(f"Selected {selected_points.shape[0]} points with 10 < kNN_percentile < {slider.value}")

# Select internal points at core
internal_mask = LC_only_points['kNN_percentile'] < 10
internal_points = LC_only_points.loc[internal_mask, ['x', 'y', 'z']].values
print(f"Selected {internal_points.shape[0]} internal points with kNN < 10")

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=selected_points[:, 0],  # y
        y=selected_points[:, 1],  # x
        z=selected_points[:, 2],  # z
        mode='markers',
        marker=dict(
            size=1,
            color='orange',
            opacity=0.5
        ),
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
)
fig.show()

# Mesh generation functions

In [ ]:
def estimate_normals(shell_points, k=40):
    k_normals = 40  # Number of neighbors to use for normal estimation

    # Build KD-tree
    nbrs = NearestNeighbors(n_neighbors=k_normals+1, algorithm='kd_tree').fit(shell_points)

    # Calculate normals using PCA of local neighborhoods
    normals = np.zeros((len(shell_points), 3))

    for i in tqdm(range(len(shell_points))):
        distances, indices = nbrs.kneighbors([shell_points[i]])
        neighbors = shell_points[indices[0]]
        
        # Center the neighborhood
        centered = neighbors - np.mean(neighbors, axis=0)
        
        # Perform PCA using SVD
        u, s, vh = np.linalg.svd(centered, full_matrices=False)
        
        # Normal is the eigenvector corresponding to the smallest eigenvalue
        normal = vh[2, :]
        
        normals[i] = normal

    # normals to unit length
    normals_normalized = normals / np.linalg.norm(normals, axis=1, keepdims=True)
    return normals_normalized

def orient_normals_with_interior(shell_points, normals, interior_points, k_interior=30, k_smooth=20):
    """
    Orient normals on a shell using interior points to determine direction.
    
    Args:
        shell_points: Nx3 array of point coordinates on the shell
        normals: Nx3 array of normal vectors (assumed to be unit length)
        interior_points: Mx3 array of interior points
        k_interior: Number of interior points to consider for each shell point
        k_smooth: Number of neighbors for smoothing consistency
        
    Returns:
        Nx3 array of consistently oriented normals
    """
    oriented_normals = normals.copy()
    n_points = len(shell_points)
    
    print(f"Orienting {n_points} normals using {len(interior_points)} interior points...")
    
    # For each shell point, find nearest interior points
    nbrs_interior = NearestNeighbors(n_neighbors=k_interior, algorithm='kd_tree').fit(interior_points)
    distances, indices = nbrs_interior.kneighbors(shell_points)
    
    # Compute vector from interior points to shell points
    inside_to_outside = np.zeros_like(shell_points)
    for i in range(n_points):
        # Get weighted average of directions from nearest interior points to this shell point
        nearby_interior = interior_points[indices[i]]
        
        # Weight by inverse distance
        weights = 1.0 / (distances[i] + 1e-10)
        weights = weights / np.sum(weights)
        
        # Weighted average interior point
        weighted_interior = np.sum(nearby_interior * weights.reshape(-1, 1), axis=0)
        
        # Vector from weighted interior to shell point
        inside_to_outside[i] = shell_points[i] - weighted_interior
    
    # Normalize direction vectors
    norms = np.linalg.norm(inside_to_outside, axis=1).reshape(-1, 1)
    inside_to_outside = np.divide(inside_to_outside, norms, out=np.zeros_like(inside_to_outside), where=norms!=0)
    
    # Dot products to determine if normals point outward
    dots = np.sum(oriented_normals * inside_to_outside, axis=1)
    
    # Flip normals that point inward
    flip_mask = dots < 0
    oriented_normals[flip_mask] *= -1
    
    print(f"Flipped {np.sum(flip_mask)} normals based on interior point orientation")
    
    # Apply local consistency smoothing
    nbrs_shell = NearestNeighbors(n_neighbors=k_smooth+1, algorithm='kd_tree').fit(shell_points)
    _, indices = nbrs_shell.kneighbors(shell_points)
    
    # Iterative smoothing
    max_iterations = 3
    for iteration in range(max_iterations):
        changes = 0
        
        for i in range(n_points):
            neighbors = indices[i, 1:]
            
            # Check if normal is consistent with neighbors
            dot_products = np.sum(oriented_normals[i] * oriented_normals[neighbors], axis=1)
            agreement_count = np.sum(dot_products > 0)
            disagreement_count = len(neighbors) - agreement_count
            
            # If more neighbors disagree than agree, flip the normal
            if disagreement_count > agreement_count:
                oriented_normals[i] *= -1
                changes += 1
        
        print(f"Iteration {iteration+1}: Flipped {changes} normals for consistency")
        if changes == 0:
            break
    
    return oriented_normals

def propagate_through_patches(shell_points, normals, patch_size=100, overlap=0.5):
    """
    Process the point cloud in overlapping patches for better local consistency.
    Useful for very large point clouds with varying density.
    
    Args:
        shell_points: Nx3 array of point coordinates
        normals: Nx3 array of normal vectors
        patch_size: Approximate number of points per patch
        overlap: Overlap factor between patches (0-1)
        
    Returns:
        Nx3 array of consistently oriented normals
    """
    from sklearn.cluster import KMeans
    
    n_points = len(shell_points)
    oriented_normals = normals.copy()
    
    # Determine number of patches based on point count and patch size
    n_patches = max(1, int(n_points / (patch_size * (1-overlap/2))))
    print(f"Dividing point cloud into {n_patches} patches")
    
    # Use K-means to create spatially coherent patches
    kmeans = KMeans(n_clusters=n_patches, random_state=0).fit(shell_points)
    labels = kmeans.labels_
    
    # Process each patch
    for patch_id in range(n_patches):
        patch_mask = labels == patch_id
        patch_points = shell_points[patch_mask]
        patch_normals = oriented_normals[patch_mask]
        
        print(f"Processing patch {patch_id+1}/{n_patches} with {len(patch_points)} points")
        
        # Find a consistent orientation within this patch
        # Use MST for local consistency
        k = min(30, max(10, len(patch_points)//10))
        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='kd_tree').fit(patch_points)
        distances, indices = nbrs.kneighbors(patch_points)
        
        # Build graph for MST
        rows = []
        cols = []
        weights = []
        
        for i in range(len(patch_points)):
            for j_idx, j in enumerate(indices[i, 1:]):
                # Weight by combination of distance and normal similarity
                dot_product = abs(np.dot(patch_normals[i], patch_normals[j]))
                weight = distances[i, j_idx+1] * (2.0 - dot_product)
                
                rows.append(i)
                cols.append(j)
                weights.append(weight)
        
        # Create graph and compute MST
        graph = csr_matrix((weights, (rows, cols)), shape=(len(patch_points), len(patch_points)))
        mst = minimum_spanning_tree(graph).tocsr()
        
        # Find most extreme point in patch (likely on boundary)
        center = np.mean(patch_points, axis=0)
        distances_to_center = np.linalg.norm(patch_points - center, axis=1)
        root = np.argmax(distances_to_center)
        
        # Propagate orientations through MST
        visited = np.zeros(len(patch_points), dtype=bool)
        
        def dfs(node):
            visited[node] = True
            
            row_indices, col_indices = mst[node].nonzero()
            neighbors = col_indices
            
            row_indices, col_indices = mst.nonzero()
            reverse_neighbors = row_indices[col_indices == node]
            
            all_neighbors = np.concatenate([neighbors, reverse_neighbors]) if len(neighbors) > 0 and len(reverse_neighbors) > 0 else (neighbors if len(neighbors) > 0 else reverse_neighbors)
            
            for neighbor in all_neighbors:
                if not visited[neighbor]:
                    dot_product = np.dot(patch_normals[node], patch_normals[neighbor])
                    if dot_product < 0:
                        patch_normals[neighbor] = -patch_normals[neighbor]
                    dfs(neighbor)
        
        dfs(root)
        
        # Update the global normals with the patch results
        oriented_normals[patch_mask] = patch_normals
    
    # Final global consistency refinement
    print("Performing final consistency refinement...")
    nbrs = NearestNeighbors(n_neighbors=15, algorithm='kd_tree').fit(shell_points)
    distances, indices = nbrs.kneighbors(shell_points)
    
    changes = 0
    for i in range(n_points):
        neighbors = indices[i, 1:10]  # Use first few neighbors only
        dot_products = np.sum(oriented_normals[i] * oriented_normals[neighbors], axis=1)
        if np.sum(dot_products < 0) > len(neighbors) / 2:
            oriented_normals[i] *= -1
            changes += 1
    
    print(f"Final refinement: Flipped {changes} normals")
    
    return oriented_normals

def orient_complex_shape_normals(shell_points, normals, interior_points=None):
    """
    Orient normals for a complex shape using interior points if available,
    otherwise use patch-based processing.
    
    Args:
        shell_points: Points on the surface shell 
        normals: Initial normal estimates
        interior_points: Optional interior points for guidance
        
    Returns:
        Consistently oriented normals
    """
    if interior_points is not None and len(interior_points) > 0:
        print(f"Using {len(interior_points)} interior points to guide orientation...")
        oriented_normals = orient_normals_with_interior(
            shell_points, 
            normals.copy(), 
            interior_points
        )
    else:
        print("No interior points provided, using patch-based orientation...")
        oriented_normals = propagate_through_patches(
            shell_points,
            normals.copy()  
        )
    
    return oriented_normals

def shrink_mesh_along_normals(mesh, distance):
    """
    Shrink a mesh by moving each vertex inward along its normal direction
    
    Parameters:
    -----------
    mesh : trimesh.Trimesh
        The input mesh to shrink
    distance : float
        The distance to move each vertex (positive value = shrink inward)
    
    Returns:
    --------
    trimesh.Trimesh
        A new shrunk mesh
    """
    # Get a copy of the mesh
    shrunk_mesh = mesh.copy()
    
    # Calculate vertex normals (average of adjacent face normals)
    vertex_normals = mesh.vertex_normals
    
    # Normalize the vertex normals to ensure consistent distance
    norm = np.linalg.norm(vertex_normals, axis=1).reshape(-1, 1)
    # Avoid division by zero
    norm[norm == 0] = 1.0
    vertex_normals_normalized = vertex_normals / norm
    
    # Move vertices inward along their normals
    shrunk_mesh.vertices = mesh.vertices - (vertex_normals_normalized * distance)
    
    # Fix any issues that might have been introduced
    shrunk_mesh.fix_normals()
    # shrunk_mesh.remove_degenerate_faces()
    
    return shrunk_mesh

In [ ]:
def seal_holes(mesh):
    """
    Seal all holes in a Trimesh mesh with flat surfaces.
    Each hole gets 1 new vertex at the centroid, fan-triangulated.
    Modifies mesh.vertices and mesh.faces in-place.
    """
    V = mesh.vertices
    F = mesh.faces

    # Step 1: count all edges
    edge_count = dict()
    for face in F:
        edges = [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]
        for a, b in edges:
            e = tuple(sorted((a, b)))
            edge_count[e] = edge_count.get(e, 0) + 1

    # Step 2: boundary edges = edges that appear only once
    boundary_edges = [e for e, count in edge_count.items() if count == 1]

    # Step 3: build loops from boundary edges
    loops = []
    edges_left = set(boundary_edges)
    while edges_left:
        loop = []
        e = edges_left.pop()
        loop.extend(e)
        while True:
            # find next edge that shares last vertex
            last = loop[-1]
            found = None
            for candidate in edges_left:
                if last in candidate:
                    found = candidate
                    break
            if found is None:
                break
            edges_left.remove(found)
            next_v = found[0] if found[1] == last else found[1]
            loop.append(next_v)
            if next_v == loop[0]:  # closed loop
                break
        # remove duplicates in loop while keeping order
        seen = set()
        loop_clean = []
        for v in loop:
            if v not in seen:
                loop_clean.append(v)
                seen.add(v)
        loops.append(loop_clean)

    # Step 4: fan-triangulate each loop
    new_vertices = []
    new_faces = []

    for loop in loops:
        coords = V[loop]
        centroid = coords.mean(axis=0)
        centroid_idx = len(V) + len(new_vertices)
        new_vertices.append(centroid)

        N = len(loop)
        for i in range(N):
            v0 = loop[i]
            v1 = loop[(i + 1) % N]  # wrap around
            new_faces.append([centroid_idx, v0, v1])

    # Step 5: update mesh
    if new_vertices:
        mesh.vertices = np.vstack([V, np.array(new_vertices)])
        mesh.faces = np.vstack([F, np.array(new_faces)])

# Initial mesh estimations

In [ ]:
shell_points = selected_points.astype(np.float32)

interior_points = internal_points.astype(np.float32)

normals_normalized = estimate_normals(shell_points, k=80)
consistently_oriented_normals = orient_complex_shape_normals(
    shell_points,
    normals_normalized.copy(),
    interior_points=interior_points
)
consistently_oriented_normals = np.asfortranarray(consistently_oriented_normals, dtype=np.float32)

vertices, faces = pcu.pointcloud_surfel_geometry(shell_points, consistently_oriented_normals, 30)

In [ ]:
consistently_oriented_normals

In [ ]:
vertices.shape, faces.shape

In [ ]:
vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices, faces, 10000)
vertices_wt.shape, faces_wt.shape

In [ ]:
# CORE MESH ESTIMATOR
shell_points = selected_points.astype(np.float32)

interior_points = internal_points.astype(np.float32)

normals_normalized = estimate_normals(shell_points, k=80)
consistently_oriented_normals = orient_complex_shape_normals(
    shell_points,
    normals_normalized.copy(),
    interior_points=interior_points
)
consistently_oriented_normals = np.asfortranarray(consistently_oriented_normals, dtype=np.float32)

vertices, faces = pcu.pointcloud_surfel_geometry(shell_points, consistently_oriented_normals, 30)

vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices, faces, 10000)
mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

while not mesh.is_watertight:
    print("Mesh is not watertight, retrying...")
    vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices_wt, faces_wt, 10000)
    mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

# Convert trimesh to Open3D
mesh_o3d = o3d.geometry.TriangleMesh(
    o3d.utility.Vector3dVector(mesh.vertices),
    o3d.utility.Vector3iVector(mesh.faces)
)

# Optional: compute vertex normals if needed
mesh_o3d.compute_vertex_normals()

# number_of_iterations: how aggressive
# lambda: smoothing strength (0 < λ < 1)
mesh_o3d = mesh_o3d.filter_smooth_simple(number_of_iterations=5)

# Convert back to trimesh
mesh_smoothed = trimesh.Trimesh(
    np.asarray(mesh_o3d.vertices),
    np.asarray(mesh_o3d.triangles)
)
new_core_mesh = mesh_smoothed

In [ ]:
new_core_mesh.volume

In [ ]:
# PERCENTILE MESH ESTIMATOR
for thresh in range(10,100,10):
    radius = 30 + (thresh - 10) * (50 - 30) / (90 - 10)
    mesh_wt_resolution = int(10000 + (thresh - 10) * (50000 - 10000) / (90 - 10))
    print(f"Thresh: {thresh}, Radius: {radius}, Mesh WT Resolution: {mesh_wt_resolution}")

mesh_dict = {}
for thresh in range(10, 100, 10):
    mask = (LC_only_points['kNN_percentile'] > 4) & (LC_only_points['kNN_percentile'] < thresh)
    selected_points = LC_only_points.loc[mask, ['x', 'y', 'z']].values.astype(np.float32)
    internal_mask = LC_only_points['kNN_percentile'] < 4
    internal_points = LC_only_points.loc[internal_mask, ['x', 'y', 'z']].values.astype(np.float32)

    if len(selected_points) < 100 or len(internal_points) < 10:
        print(f"Skipping threshold {thresh}: not enough points.")
        continue

    normals_normalized = estimate_normals(selected_points, k=80)
    consistently_oriented_normals = orient_complex_shape_normals(
        selected_points,
        normals_normalized.copy(),
        interior_points=internal_points
    )
    consistently_oriented_normals = np.asfortranarray(consistently_oriented_normals, dtype=np.float32)

    # dynamic mesh resolution: linearly map thresh=10 -> 10000, thresh=90 -> 50000
    mesh_wt_resolution = int(10000 + (thresh - 10) * (50000 - 10000) / (90 - 10))

    # Linearly interpolate radius: thresh=10 -> 30, thresh=90 -> 40
    radius = 30 + (thresh - 10) * (50 - 30) / (90 - 10)
    vertices, faces = pcu.pointcloud_surfel_geometry(selected_points, consistently_oriented_normals, radius)
    vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices, faces, mesh_wt_resolution)
    mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

    while not mesh.is_watertight:
        print(f"Mesh is not watertight at threshold {thresh}, retrying...")
        vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices_wt, faces_wt, mesh_wt_resolution)
        mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

    # Convert trimesh to Open3D
    mesh_o3d = o3d.geometry.TriangleMesh(
        o3d.utility.Vector3dVector(mesh.vertices),
        o3d.utility.Vector3iVector(mesh.faces)
    )

    # Optional: compute vertex normals if needed
    mesh_o3d.compute_vertex_normals()

    # number_of_iterations: how aggressive
    # lambda: smoothing strength (0 < λ < 1)
    mesh_o3d = mesh_o3d.filter_smooth_simple(number_of_iterations=5)

    # Convert back to trimesh
    mesh_smoothed = trimesh.Trimesh(
        np.asarray(mesh_o3d.vertices),
        np.asarray(mesh_o3d.triangles)
    )
    mesh_dict[thresh] = mesh_smoothed


In [ ]:
fig = go.Figure()

# Sort initial mesh estimates keys in descending order (90 to 10)
mesh_keys = sorted(mesh_dict.keys(), reverse=True)
colors = ['royalblue', 'blue', 'darkblue', 'lightseagreen', 'seagreen', 'darkgreen', 
          'orange', 'darkorange', 'darkred']

for i, thresh in enumerate(mesh_keys):
    mesh = mesh_dict[thresh]
    color = colors[i % len(colors)]
    fig.add_trace(
        go.Mesh3d(
            x=mesh.vertices[:, 0],
            y=mesh.vertices[:, 1],
            z=mesh.vertices[:, 2],
            i=mesh.faces[:, 0],
            j=mesh.faces[:, 1],
            k=mesh.faces[:, 2],
            opacity=0.15,
            color=color,
            name=f"Mesh {thresh}"
        )
    )

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title='Overlay of Meshes from 90 to 10'
)
fig.show()

# manual mesh repair

Initial mesh estimates generated by surfel geometry include degenerate faces, holes, caverns, tunnels and other irregularities that need to be repaired. Removal of these objects proceeded separately for each individual mesh given the distinct challenges posed by distinct characteristics for each. The steps for such manual refinements are included in this section.

In [ ]:
# MESH percentile to operate on, else None to operate on the core mesh at 67th percentile
thresh = None

In [ ]:
# VISUALIZE EXTREME SURFACE CURVATURES

if thresh is not None:
    test_mesh = mesh_dict[thresh]
else:
    test_mesh = new_core_mesh

test_vertices = test_mesh.vertices
test_faces = test_mesh.faces

L = trimesh.smoothing.laplacian_calculation(test_mesh, equal_weight=False)
H = np.linalg.norm(L.dot(test_mesh.vertices), axis=1)
sign = np.sign(np.einsum('ij,ij->i', L.dot(test_mesh.vertices), test_mesh.vertex_normals))
signed_curvature = H * sign

fig = go.Figure()

# Add mesh surface
fig.add_trace(
    go.Mesh3d(
        x=test_vertices[:, 0],
        y=test_vertices[:, 1],
        z=test_vertices[:, 2],
        i=test_faces[:, 0],
        j=test_faces[:, 1],
        k=test_faces[:, 2],
        opacity=0.3,
        color='lightgray',
        name='Mesh'
    )
)

# Add scatter of vertices color coded by signed_curvature
fig.add_trace(
    go.Scatter3d(
        x=test_vertices[:, 0],
        y=test_vertices[:, 1],
        z=test_vertices[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=signed_curvature,
            colorscale='RdBu',
            colorbar=dict(title='Signed Curvature'),
            opacity=0.5
        ),
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
)
fig.show()


In [ ]:
# --- Voxelize the mesh ---
pitch = 3  # smaller = finer, adjust to your mesh size
voxelized = test_mesh.voxelized(pitch=pitch).fill()
voxel_matrix = voxelized.matrix
voxel_coords = voxelized.points
print(f"Voxel matrix shape: {voxel_matrix.shape}, pitch: {pitch}")

# --- Compute distance transform ---
# distance of each inside voxel to nearest outside voxel
# distance = ndimage.distance_transform_edt(voxel_matrix)

max_distance = 5  # voxels, adjust based on results

# initialize distance array
distance = np.zeros_like(voxel_matrix, dtype=np.uint8)

# frontier = surface voxels
surface = ndimage.binary_erosion(voxel_matrix) ^ voxel_matrix
frontier = surface.copy()

for d in tqdm(range(1, max_distance+1), desc="Computing distance transform"):
    distance[frontier] = d
    # grow frontier one voxel into interior
    frontier = ndimage.binary_dilation(frontier) & voxel_matrix & (distance == 0)

# label anything beyond max_distance as max_distance + 1
distance[distance == 0] = max_distance + 1

print("Computed distance transform.")

# --- Get coordinates of voxel centers ---
tree = cKDTree(voxel_coords)

# Map mesh vertices to nearest filled voxel
_, nearest_idx = tree.query(test_mesh.vertices, k=1)

# Distances ONLY for filled voxels
distance_flat = distance[voxel_matrix]

vertex_distance = distance_flat[nearest_idx]


In [ ]:
# VISUALIZE CAVERNS DESCENDING FROM MESH SURFACE

fig = go.Figure()

# Add mesh surface in gray
fig.add_trace(
    go.Mesh3d(
        x=test_mesh.vertices[:, 0],
        y=test_mesh.vertices[:, 1],
        z=test_mesh.vertices[:, 2],
        i=test_mesh.faces[:, 0],
        j=test_mesh.faces[:, 1],
        k=test_mesh.faces[:, 2],
        color='lightgray',
        opacity=0.5,
        name='Mesh'
    )
)

# Add scatter of vertices colored by vertex_distance, with edge color set to none, vmin=0, vmax=5
fig.add_trace(
    go.Scatter3d(
        x=test_mesh.vertices[:, 0],
        y=test_mesh.vertices[:, 1],
        z=test_mesh.vertices[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color=vertex_distance,
            colorscale='BuPu',
            cmin=0,
            cmax=4,
            colorbar=dict(title='Distance to Surface'),
            opacity=0.5,
            line=dict(width=0)  # edge color = none
        ),
        name='Vertices (Distance)'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title='Mesh and Vertices Colored by Distance to Surface'
)
fig.show()


In [ ]:
test_vertices = test_mesh.vertices
test_faces = test_mesh.faces

# Remove vertices where vertex_distance > 2
keep_mask = vertex_distance <= 2
filtered_vertices = test_vertices[keep_mask]

# Map old vertex indices to new indices
old_to_new = -np.ones(len(test_vertices), dtype=int)
old_to_new[keep_mask] = np.arange(np.sum(keep_mask))

# Filter faces: keep only faces where all three vertices are kept
face_mask = keep_mask[test_faces].all(axis=1)
filtered_faces = test_faces[face_mask]
# Remap face indices to new vertex indices
filtered_faces = old_to_new[filtered_faces]

In [ ]:
fig = go.Figure()

# PLOT MESH AFTER VERTEX REMOVEL

fig.add_trace(
    go.Mesh3d(
        x=filtered_vertices[:, 0],
        y=filtered_vertices[:, 1],
        z=filtered_vertices[:, 2],
        i=filtered_faces[:, 0],
        j=filtered_faces[:, 1],
        k=filtered_faces[:, 2],
        color='purple',
        opacity=0.4,
        name='Original Mesh'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title='IRREGUARITIES REMOVED'
)
fig.show()

In [ ]:
sealed_mesh = trimesh.Trimesh(vertices=filtered_vertices, faces=filtered_faces)

trimesh.repair.fix_winding(sealed_mesh)
seal_holes(sealed_mesh)
trimesh.repair.fix_winding(sealed_mesh)
trimesh.repair.fill_holes(sealed_mesh)
trimesh.repair.fix_normals(sealed_mesh, multibody=True)
sealed_mesh.is_watertight

In [ ]:
sealed_mesh.vertices.shape, sealed_mesh.faces.shape

In [ ]:
sealed_mesh.volume

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Mesh3d(
        x=sealed_mesh.vertices[:, 0],
        y=sealed_mesh.vertices[:, 1],
        z=sealed_mesh.vertices[:, 2],
        i=sealed_mesh.faces[:, 0],
        j=sealed_mesh.faces[:, 1],
        k=sealed_mesh.faces[:, 2],
        opacity=0.5,
        color='orange',
        name='Sealed Mesh'
    )
)

fig.add_trace(
    go.Scatter3d(
        x=test_mesh.vertices[:, 0],
        y=test_mesh.vertices[:, 1],
        z=test_mesh.vertices[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color='purple',
            opacity=0.3
        ),
        name='Original Filtered Mesh Vertices'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title='Sealed Mesh Visualization'
)
fig.show()

In [ ]:
broken = trimesh.repair.broken_faces(sealed_mesh)

b = np.asarray(broken)

if b.size == 0:
    print("No broken faces.")
else:
    all_idx = np.arange(len(sealed_mesh.faces))
    remove_idx = b.astype(int)
    keep_idx = np.setdiff1d(all_idx, remove_idx, assume_unique=True)
    removed_count = len(remove_idx)

    # apply the explicit keep list and cleanup
    sealed_mesh.faces = sealed_mesh.faces[keep_idx]
    sealed_mesh.remove_unreferenced_vertices()
    print(f"Removed {removed_count} broken faces, remaining faces: {len(sealed_mesh.faces)}")

    # Remove faces that have solitary (non-shared) edges

    # Count all edges
    edge_count = defaultdict(int)
    for face in sealed_mesh.faces:
        edges = [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]
        for a, b in edges:
            e = tuple(sorted((a, b)))
            edge_count[e] += 1

    # Find all faces that have at least one solitary edge
    solitary_face_mask = []
    for face in sealed_mesh.faces:
        edges = [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]
        has_solitary = any(edge_count[tuple(sorted((a, b)))] == 1 for a, b in edges)
        solitary_face_mask.append(not has_solitary)

    solitary_face_mask = np.array(solitary_face_mask)
    n_removed = np.sum(~solitary_face_mask)
    sealed_mesh.faces = sealed_mesh.faces[solitary_face_mask]
    sealed_mesh.remove_unreferenced_vertices()
    print(f"Removed {n_removed} faces with solitary edges, remaining faces: {len(sealed_mesh.faces)}")


In [ ]:
# uncomment IF TEST VERTICES ARE NOT ALIGNED WITH THE REPAIRED MESH FACES

# sealed_mesh = shrink_mesh_along_normals(sealed_mesh, distance=5)

In [ ]:
# uncomment IF HOLES REMAIN

# trimesh.repair.fix_winding(sealed_mesh)
# seal_holes(sealed_mesh)
# trimesh.repair.fix_winding(sealed_mesh)
# trimesh.repair.fill_holes(sealed_mesh)
# trimesh.repair.fix_normals(sealed_mesh, multibody=True)
# sealed_mesh.is_watertight

In [ ]:
output_dir = '/root/capsule/results/percentile_meshes'
os.makedirs(output_dir, exist_ok=True)

if thresh is None:
    output_filename = 'new_core_mesh.obj'
else:
    output_filename = f'percentile_{thresh}.obj'

output_path = os.path.join(output_dir, output_filename)
sealed_mesh.export(output_path)
print(f"Saved sealed_mesh to {output_path}")

# tmp test

In [ ]:
shared_mesh = trimesh.load('/root/capsule/data/LC_percentile_meshes/new_core_mesh.obj')

In [ ]:
np.abs(sealed_mesh.volume - shared_mesh.volume) / shared_mesh.volume * 100

In [ ]:
sealed_mesh.vertices.shape, sealed_mesh.faces.shape, shared_mesh.vertices.shape, shared_mesh.faces.shape

In [ ]:
# Check if sealed_mesh vertices are present in shared_mesh
tree_shared = cKDTree(shared_mesh.vertices)
dists_to_shared, _ = tree_shared.query(sealed_mesh.vertices, k=1)

print(f"sealed_mesh vertices: {len(sealed_mesh.vertices)}")
print(f"shared_mesh vertices: {len(shared_mesh.vertices)}")
print(f"\nNearest-neighbor distances (sealed -> shared):")
print(f"  Min:    {dists_to_shared.min():.6f}")
print(f"  Max:    {dists_to_shared.max():.6f}")
print(f"  Mean:   {dists_to_shared.mean():.6f}")
print(f"  Median: {np.median(dists_to_shared):.6f}")

for tol in [0.0, 1e-6, 1e-3, 0.1, 1.0, 10.0]:
    n_match = np.sum(dists_to_shared <= tol)
    print(f"  Vertices within {tol}: {n_match} / {len(sealed_mesh.vertices)} ({100*n_match/len(sealed_mesh.vertices):.2f}%)")

In [ ]:
# Symmetric Hausdorff distance between the two mesh SURFACES.
#
# Shortcut: for each query point, ask trimesh for the closest point on the
# *surface* (any point inside any triangle) of the other mesh, not the closest
# vertex. trimesh.proximity.closest_point() does this exactly via a triangle
# AABB tree, so we never enumerate vertex pairs.
#
# To approximate the surface well we sample points uniformly over each mesh
# (area-weighted) and also include the vertices to catch sharp features.

n_samples = 250000

# Uniform surface samples
pts_sealed, _ = trimesh.sample.sample_surface(sealed_mesh, n_samples)
pts_shared, _ = trimesh.sample.sample_surface(shared_mesh, n_samples)

# Add vertices to make sure extrema are not missed
pts_sealed = np.vstack([pts_sealed, sealed_mesh.vertices])
pts_shared = np.vstack([pts_shared, shared_mesh.vertices])

# Distance from sealed-mesh surface points to the shared-mesh SURFACE
_, d_s2sh, _ = trimesh.proximity.closest_point(shared_mesh, pts_sealed)
# Distance from shared-mesh surface points to the sealed-mesh SURFACE
_, d_sh2s, _ = trimesh.proximity.closest_point(sealed_mesh, pts_shared)

h_s2sh = float(d_s2sh.max())
h_sh2s = float(d_sh2s.max())
hausdorff = max(h_s2sh, h_sh2s)

print(f"max(sealed -> shared surface): {h_s2sh:.4f}")
print(f"max(shared -> sealed surface): {h_sh2s:.4f}")
print(f"Symmetric Hausdorff distance:  {hausdorff:.4f}")

print("\nSurface-to-surface distance stats (sealed -> shared):")
print(f"  mean   {d_s2sh.mean():.4f}")
print(f"  median {np.median(d_s2sh):.4f}")
print(f"  95%    {np.percentile(d_s2sh, 95):.4f}")
print(f"  99%    {np.percentile(d_s2sh, 99):.4f}")

print("\nSurface-to-surface distance stats (shared -> sealed):")
print(f"  mean   {d_sh2s.mean():.4f}")
print(f"  median {np.median(d_sh2s):.4f}")
print(f"  95%    {np.percentile(d_sh2s, 95):.4f}")
print(f"  99%    {np.percentile(d_sh2s, 99):.4f}")

In [ ]:
# Visualize pts_sealed colored by distance to the shared_mesh surface.
# d_s2sh was computed in the previous cell alongside pts_sealed.

fig = go.Figure()

# Shared mesh as a faint reference surface
fig.add_trace(go.Mesh3d(
    x=shared_mesh.vertices[:, 0],
    y=shared_mesh.vertices[:, 1],
    z=shared_mesh.vertices[:, 2],
    i=shared_mesh.faces[:, 0],
    j=shared_mesh.faces[:, 1],
    k=shared_mesh.faces[:, 2],
    color='lightgray',
    opacity=0.2,
    name='shared_mesh',
    showscale=False,
))

# Sampled sealed-mesh surface points, colored by distance to shared_mesh surface
fig.add_trace(go.Scatter3d(
    x=pts_sealed[:, 0],
    y=pts_sealed[:, 1],
    z=pts_sealed[:, 2],
    mode='markers',
    marker=dict(
        size=2,
        color=d_s2sh,
        colorscale='Inferno',
        cmin=0,
        cmax=1.0, #float(np.percentile(d_s2sh, 99.9)),  # clip extremes for visibility
        colorbar=dict(title='dist to shared surface'),
        opacity=0.8,
        line=dict(width=0),
    ),
    name='pts_sealed',
))

fig.update_layout(
    title=f'pts_sealed colored by distance to shared_mesh '
          f'(max={d_s2sh.max():.2f}, mean={d_s2sh.mean():.2f})',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data',
    ),
)
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Mesh3d(
        x=shared_mesh.vertices[:, 0],
        y=shared_mesh.vertices[:, 1],
        z=shared_mesh.vertices[:, 2],
        i=shared_mesh.faces[:, 0],
        j=shared_mesh.faces[:, 1],
        k=shared_mesh.faces[:, 2],
        color='royalblue',
        opacity=0.1,
        name='shared_mesh'
    )
)

fig.add_trace(
    go.Mesh3d(
        x=sealed_mesh.vertices[:, 0],
        y=sealed_mesh.vertices[:, 1],
        z=sealed_mesh.vertices[:, 2],
        i=sealed_mesh.faces[:, 0],
        j=sealed_mesh.faces[:, 1],
        k=sealed_mesh.faces[:, 2],
        color='crimson',
        opacity=0.1,
        name='sealed_mesh'
    )
)

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data'
    ),
    title='shared_mesh (blue) vs sealed_mesh (red)'
)
fig.show()

In [ ]:
# Are any LC points inside one mesh but not the other?
coords = LC_only_points[['x', 'y', 'z']].values

batch_size = 5000
n = len(coords)
in_sealed = np.zeros(n, dtype=bool)
in_shared = np.zeros(n, dtype=bool)

for start in tqdm(range(0, n, batch_size), desc='containment'):
    end = min(start + batch_size, n)
    pts = coords[start:end]
    in_sealed[start:end] = sealed_mesh.contains(pts)
    in_shared[start:end] = shared_mesh.contains(pts)

only_sealed = in_sealed & ~in_shared
only_shared = ~in_sealed & in_shared
disagree = only_sealed | only_shared

print(f"Total LC points:     {n}")
print(f"Inside sealed_mesh:  {int(in_sealed.sum())}")
print(f"Inside shared_mesh:  {int(in_shared.sum())}")
print(f"Inside both:         {int((in_sealed & in_shared).sum())}")
print(f"Only in sealed:      {int(only_sealed.sum())}")
print(f"Only in shared:      {int(only_shared.sum())}")
print(f"Disagree total:      {int(disagree.sum())} "
      f"({100*disagree.sum()/n:.3f}% of LC points)")

# Visualize the disagreement points relative to both meshes
fig = go.Figure()

for m, name, color in [(sealed_mesh, 'sealed_mesh', 'lightcoral'),
                       (shared_mesh, 'shared_mesh', 'lightsteelblue')]:
    fig.add_trace(go.Mesh3d(
        x=m.vertices[:, 0], y=m.vertices[:, 1], z=m.vertices[:, 2],
        i=m.faces[:, 0], j=m.faces[:, 1], k=m.faces[:, 2],
        color=color, opacity=0.15, name=name, showlegend=True,
    ))

pts_only_sealed = coords[only_sealed]
pts_only_shared = coords[only_shared]

if len(pts_only_sealed) > 0:
    fig.add_trace(go.Scatter3d(
        x=pts_only_sealed[:, 0], y=pts_only_sealed[:, 1], z=pts_only_sealed[:, 2],
        mode='markers',
        marker=dict(size=3, color='crimson', opacity=0.85),
        name=f'only in sealed ({len(pts_only_sealed)})',
    ))

if len(pts_only_shared) > 0:
    fig.add_trace(go.Scatter3d(
        x=pts_only_shared[:, 0], y=pts_only_shared[:, 1], z=pts_only_shared[:, 2],
        mode='markers',
        marker=dict(size=3, color='royalblue', opacity=0.85),
        name=f'only in shared ({len(pts_only_shared)})',
    ))

fig.update_layout(
    title='LC points with differential mesh membership',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z', aspectmode='data'),
)
fig.show()

# Load from save

In [ ]:
mesh_dir = '/root/capsule/data/LC_percentile_meshes'
mesh_files = [f for f in os.listdir(mesh_dir) if f.startswith('percentile') and f.endswith('.obj')]
percentile_meshes = {}

for fname in mesh_files:
    mesh_path = os.path.join(mesh_dir, fname)
    mesh = trimesh.load(mesh_path)
    fname = os.path.splitext(fname)[0]
    key = int(fname.split('_')[1])  # Extract threshold from filename
    percentile_meshes[key] = mesh

print(f"Loaded {len(percentile_meshes)} percentile_meshes from {mesh_dir}")

In [ ]:
# Check if watertight
for fname, mesh in percentile_meshes.items():
    print(f"{fname}: watertight = {mesh.is_watertight}")

In [ ]:
fig = go.Figure()

# Sort percentile_meshes keys
mesh_keys = sorted(percentile_meshes.keys(), reverse=True)
colors = ['royalblue', 'blue', 'darkblue', 'lightseagreen', 'seagreen', 'darkgreen', 
          'orange', 'darkorange', 'darkred']

for i, thresh in enumerate(mesh_keys):
    mesh = percentile_meshes[thresh]
    color = colors[i % len(colors)]
    fig.add_trace(
        go.Mesh3d(
            x=mesh.vertices[:, 0],
            y=mesh.vertices[:, 1],
            z=mesh.vertices[:, 2],
            i=mesh.faces[:, 0],
            j=mesh.faces[:, 1],
            k=mesh.faces[:, 2],
            opacity=0.15,
            color=color,
            name=f"Mesh {thresh}"
        )
    )

fig.update_layout(
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    title='Overlay of Meshes from 90 to 10'
)
fig.show()

In [ ]:
new_core_mesh = trimesh.load(os.path.join(mesh_dir, 'new_core_mesh.obj'))

In [ ]:
LC_only_points = pd.read_csv(os.path.join(mesh_dir, 'LC_points.csv'))

# COUNTS PER ROI

In [ ]:
LC_only_points['in_new_core_mesh'] = 0
coords = LC_only_points[['x', 'y', 'z']].values
batch_size = 1000
n = len(coords)

for start in tqdm(range(0, n, batch_size)):
    end = min(start + batch_size, n)
    batch_idx = np.arange(start, end)
    patch_pts = coords[batch_idx]
    inside = new_core_mesh.contains(patch_pts)
    inside_idx = batch_idx[inside]
    if inside_idx.size:
        LC_only_points.loc[LC_only_points.index[inside_idx], 'in_new_core_mesh'] = 1
    print(f"Batch {start}-{end}: {np.sum(inside)} inside new core mesh")

# Count number of points in core mesh for each hemisphere
counts_per_file = LC_only_points.groupby(['file', 'reflected'])['in_new_core_mesh'].sum()

# Compute mean and standard deviation
mean_count = counts_per_file.mean()
std_count = counts_per_file.std()

print(f"Average number of points in core mesh across all hemispheres: {mean_count:.2f} ± {std_count:.2f}")

In [ ]:
LC_only_points.to_csv('/root/capsule/results/percentile_meshes/LC_points.csv', index=False)

In [ ]:
print(f"Percentage of LC_only_points inside new core mesh: {LC_only_points['in_new_core_mesh'].mean() * 100:.2f}%")

In [ ]:
print(f"Volume of new_core_mesh: {new_core_mesh.volume / 1e9:.6f} mm³")

In [ ]:
# Add columns for each percentile mesh
mesh_keys = sorted(percentile_meshes.keys())

for key in mesh_keys:
    col_name = f'in_{key}'
    LC_only_points[col_name] = 0
    
for key in mesh_keys:
    mesh = percentile_meshes[key]
    print(f"{key}: volume = {mesh.volume:.2f}")
    
# coords = LC_only_points[['y', 'x', 'z']].values
coords = LC_only_points[['x', 'y', 'z']].values
assigned_mask = np.zeros(len(LC_only_points), dtype=bool)
batch_size = 1000 

for i, key in enumerate(sorted(mesh_keys)):
    mesh = percentile_meshes[key]
    col_name = f'in_{key}'
    idx_to_check = np.where(~assigned_mask)[0]
    pts_to_check = coords[idx_to_check]
    bounds = mesh.bounds
    in_bbox = np.all((pts_to_check >= bounds[0]) & (pts_to_check <= bounds[1]), axis=1)
    bbox_idx = idx_to_check[in_bbox]
    pts_in_bbox = coords[bbox_idx]
    print(f"Checking {len(pts_in_bbox)} points inside bbox of {key}")

    # Process in batches
    for start in range(0, len(bbox_idx), batch_size):
        end = start + batch_size
        batch_idx = bbox_idx[start:end]
        batch_pts = coords[batch_idx]
        inside = mesh.contains(batch_pts)
        print(f"Batch {start}-{end}: {np.sum(inside)} inside")
        for higher_key in mesh_keys[i:]:
            higher_col = f'in_{higher_key}'
            LC_only_points.loc[LC_only_points.index[batch_idx[inside]], higher_col] = 1
        assigned_mask[batch_idx[inside]] = True

In [ ]:
files = LC_only_points['file'].unique()
percentile_cols = [col for col in LC_only_points.columns if 'in_' in col]
files = np.sort(files)
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(12, 6))

x_base = np.arange(len(percentile_cols)) * 2 + 1
for file_idx, file in enumerate(files):
    color = colors[file_idx % len(colors)]
    for i, col in enumerate(percentile_cols):
        group = LC_only_points[LC_only_points['file'] == file].groupby('reflected')[col].sum()
        y0 = group.get(0, 0)
        y1 = group.get(1, 0)
        ax.plot([x_base[i], x_base[i]+1], [y0, y1], marker='o', color=color, label=f'{file}' if i == 0 else "", alpha=0.7)

ax.set_xticks(x_base + 0.5)
ax.set_xticklabels([col.replace('in_', '') for col in percentile_cols], rotation=45)
ax.set_ylabel('Sum')
ax.set_xlabel('Percentile Mesh')
ax.set_title('Sum of Points in Each Percentile Mesh by Hemisphere and Brain')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


# FIG GEN

In [ ]:
# Filter to core mesh points only
core_pts = LC_only_points[LC_only_points['in_new_core_mesh'] == 1].copy()
coords = core_pts[['x', 'y', 'z']].values
knn_vals = core_pts['kNN_percentile'].values

pixel_size = 20  # microns per pixel

def min_projection_heatmap(coords, values, axis, pixel_size):
    """
    Project points onto the plane perpendicular to `axis` (0=x, 1=y, 2=z).
    For each pixel, take the minimum kNN value of all points landing in that bin.
    Returns: 2D array (min projection), extent for imshow.
    """
    # The two axes we project onto
    ax0, ax1 = [a for a in range(3) if a != axis]
    
    u = coords[:, ax0]
    v = coords[:, ax1]
    
    u_min, u_max = u.min(), u.max()
    v_min, v_max = v.min(), v.max()
    
    n_u = int(np.ceil((u_max - u_min) / pixel_size)) + 1
    n_v = int(np.ceil((v_max - v_min) / pixel_size)) + 1
    
    # Bin indices
    u_idx = ((u - u_min) / pixel_size).astype(int)
    v_idx = ((v - v_min) / pixel_size).astype(int)
    
    # Clamp
    u_idx = np.clip(u_idx, 0, n_u - 1)
    v_idx = np.clip(v_idx, 0, n_v - 1)
    
    # Initialize with NaN
    heatmap = np.full((n_v, n_u), np.nan)
    
    # Fill with minimum kNN per pixel
    for i in range(len(values)):
        ui, vi, val = u_idx[i], v_idx[i], values[i]
        if np.isnan(heatmap[vi, ui]) or val < heatmap[vi, ui]:
            heatmap[vi, ui] = val
    
    extent = [u_min, u_min + n_u * pixel_size, v_min + n_v * pixel_size, v_min]
    return heatmap, extent, ax0, ax1

axis_labels = {0: 'x', 1: 'y', 2: 'z'}
proj_names = {0: 'YZ (project along X)', 1: 'XZ (project along Y)', 2: 'XY (project along Z)'}

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

vmin_global = np.nanpercentile(knn_vals, 1)
vmax_global = np.nanpercentile(knn_vals, 99)

for i, proj_axis in enumerate([0, 1, 2]):
    heatmap, extent, ax0, ax1 = min_projection_heatmap(coords, knn_vals, proj_axis, pixel_size)
    
    im = axes[i].imshow(
        heatmap,
        extent=extent,
        origin='upper',
        aspect='equal',
        cmap='viridis',
        vmin=vmin_global,
        vmax=vmax_global,
        interpolation='nearest'
    )
    axes[i].set_xlabel(f'{axis_labels[ax0]} (µm)')
    axes[i].set_ylabel(f'{axis_labels[ax1]} (µm)')
    axes[i].set_title(f'Min kNN — {proj_names[proj_axis]}')
    plt.colorbar(im, ax=axes[i], label='Min kNN', shrink=0.8)

plt.suptitle('Minimum Projection of kNN (core mesh points only, 20 µm/px)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/root/capsule/results/kNN_min_projection_heatmaps.png', format='png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Core mesh points used: {len(core_pts)}")
print(f"Pixel size: {pixel_size} µm")

In [ ]:
def get_scene_layout():
    """Return standard scene layout"""
    return dict(
        xaxis_title='',
        yaxis_title='',
        zaxis_title='',
        aspectmode='data',
        xaxis=dict(showbackground=False, gridcolor="white", showticklabels=False,
                   showspikes=False, showaxeslabels=False, visible=False),
        yaxis=dict(showbackground=False, gridcolor="white", showticklabels=False,
                   showspikes=False, showaxeslabels=False, visible=False),
        zaxis=dict(showbackground=False, gridcolor="white", showticklabels=False,
                   showspikes=False, showaxeslabels=False, visible=False),
        bgcolor='rgba(0,0,0,0)',
        camera=dict(projection=dict(type='orthographic'))
    )

def add_mesh_trace(fig, mesh, color, opacity, name, reflect=False, reflect_axis=0, reflect_point=5700):
    """Add a mesh to figure, optionally reflected across midline"""
    vertices = mesh.vertices.copy()
    faces = mesh.faces
    i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]
    
    if reflect:
        vertices[:, reflect_axis] = 2 * reflect_point - vertices[:, reflect_axis]
    
    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=i, j=j, k=k,
        color=color, opacity=opacity, name=name
    ))

def add_brain_mesh_trace(fig, brain_mesh):
    """Add brain mesh"""
    vertices = brain_mesh.vertices
    faces = brain_mesh.faces
    i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]
    fig.add_trace(go.Mesh3d(
        x=vertices[:, 2], y=vertices[:, 0], z=vertices[:, 1],
        i=i, j=j, k=k,
        color='gray', opacity=0.1
    ))

def finalize_and_show(fig, save_path=None):
    """Finalize figure layout and optionally save."""
    fig.update_layout(
        showlegend=False,
        scene=get_scene_layout(),
        dragmode='orbit',
        scene_camera=dict(
            eye=dict(x=-1.1, y=-0.9, z=-0.7),
            center=dict(x=0, y=0, z=0),
            up=dict(x=0, y=0, z=-1)
        )
    )
    
    if save_path:
        html_path = save_path
        fig.write_html(html_path, include_plotlyjs=True)
        print(f"Saved figure to {html_path}")
    
    fig.show()

# 90th percentile + core mesh
brain_mesh = trimesh.load('/root/capsule/data/ccf_meshes/mcc/annotation/ccf_2017/structure_meshes/structure_997.obj')
colors = ['royalblue', 'blue', 'darkblue', 'lightseagreen', 'seagreen', 'darkgreen', 
          'orange', 'darkorange', 'darkred']

fig = go.Figure()
add_mesh_trace(fig, percentile_meshes[90], 'royalblue', 0.3, '90th Percentile Mesh')
add_mesh_trace(fig, percentile_meshes[90], 'royalblue', 0.3, '90th Percentile Mesh (Reflected)', reflect=True)
add_mesh_trace(fig, new_core_mesh, 'red', 0.6, 'LC Core Mesh')
add_mesh_trace(fig, new_core_mesh, 'red', 0.6, 'LC Core Mesh (Reflected)', reflect=True)
add_brain_mesh_trace(fig, brain_mesh)

finalize_and_show(fig, save_path='/root/capsule/results/percentile_90_plus_core.html')

# All percentile meshes
fig = go.Figure()
for i, mesh_key in enumerate(reversed(mesh_keys)):
    percentile = int(mesh_key)
    opacity = 0.5 - ((len(mesh_keys) - 1 - i) * 0.05)
    color = colors[i % len(colors)]
    
    add_mesh_trace(fig, percentile_meshes[mesh_key], color, opacity, 
                   f'{percentile}th Percentile Mesh', reflect=True)
    add_mesh_trace(fig, percentile_meshes[mesh_key], color, opacity, 
                   f'{percentile}th Percentile Mesh')

add_brain_mesh_trace(fig, brain_mesh)

finalize_and_show(fig, save_path='/root/capsule/results/all_percentiles.html')

# Points colored by core mesh membership
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 0, 'x'],
    y=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 0, 'y'],
    z=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 0, 'z'],
    mode='markers',
    marker=dict(size=1, color='royalblue', opacity=0.2),
    name='Outside Core'
))

fig.add_trace(go.Scatter3d(
    x=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 1, 'x'],
    y=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 1, 'y'],
    z=LC_only_points.loc[LC_only_points['in_new_core_mesh'] == 1, 'z'],
    mode='markers',
    marker=dict(size=1, color='crimson', opacity=0.2),
    name='Inside Core'
))

add_mesh_trace(fig, percentile_meshes[90], 'royalblue', 0.3, '90th Percentile Mesh', reflect=True)
add_mesh_trace(fig, new_core_mesh, 'red', 0.6, 'LC Core Mesh', reflect=True)
add_brain_mesh_trace(fig, brain_mesh)

finalize_and_show(fig, save_path='/root/capsule/results/LC_points_red_in_core.html')

# POINTS ON RAW IMAGES

In [ ]:
def plot_max_proj_with_points(zarr_vol, pts, ap_range=(3100, 3200), vmax=1500, output_dir='/root/capsule/results'):

    # Extract and project
    zarr_arr = zarr_vol[0, 0, :, ap_range[0]:ap_range[1], :]
    max_proj_axis1 = zarr_arr.max(axis=1)
    print(f"Zarr volume slice: shape {zarr_arr.shape}")
    print(f"Max projection: shape {max_proj_axis1.shape}")
    
    # Filter points in range
    mask = (pts[:, 1] >= ap_range[0]) & (pts[:, 1] < ap_range[1])
    pts_in_range = pts[mask]
    print(f"{pts_in_range.shape[0]} points in AP range [{ap_range[0]}, {ap_range[1]})")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(15, 15))
    im = ax.imshow(max_proj_axis1, cmap='gray_r', vmin=0, vmax=vmax)
    
    if pts_in_range.shape[0] > 0:
        scatter = ax.scatter(pts_in_range[:, 2], pts_in_range[:, 0], 
                            s=2, c='royalblue', alpha=0.6, edgecolors='none')
        ax.legend([scatter], [f'{pts_in_range.shape[0]} LC points'], loc='upper left')
    
    ax.set_title(f'Max Projection (AP {ap_range[0]}:{ap_range[1]})')
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Z (pixels)')
    plt.tight_layout()
    plt.show()
    
    # Save TIFF
    output_path = os.path.join(output_dir, f'max_proj_AP_{ap_range[0]:04d}_{ap_range[1]:04d}.tiff')
    os.makedirs(output_dir, exist_ok=True)
    tifffile.imwrite(output_path, max_proj_axis1.astype(np.uint16))
    
    return max_proj_axis1

pts_path = '/root/capsule/data/LC_H2B_trailmap_probabilities_and_point_calls/segmentation_and_quantification/SmartSPIM_807324_2025-08-25_11-34-40_stitched_2025-10-23_17-35-23/points_raw_807324.npy'
pts = np.load(pts_path)
print(f"Loaded points: shape {pts.shape}, dtype {pts.dtype}")

zarr_path = '/root/capsule/data/SmartSPIM_807324_2025-08-25_11-34-40_stitched_2025-10-23_17-35-23/image_tile_fusing/OMEZarr/Ex_488_Em_525.zarr'
zarr_data = zarr.open(zarr_path, mode='r')

print(f"Available arrays: {list(zarr_data.keys())}")

zarr_vol = zarr_data['1']
print(f"Level '1': shape {zarr_vol.shape}")

ap_ranges = [(ap, ap + 250) for ap in range(3000, 3100, 100)]

output_dir = '/root/capsule/results'
projections = {}

for ap_start, ap_end in ap_ranges:
    proj = plot_max_proj_with_points(
        zarr_vol, 
        pts, 
        ap_range=(ap_start, ap_end), 
        vmax=10000,
        output_dir=output_dir
    )
    projections[(ap_start, ap_end)] = proj

# Points with PC2 hists

In [ ]:
y_planes = np.arange(10000, 11001, 100)
min_x, max_x = LC_only_points['x'].min(), LC_only_points['x'].max()
min_z, max_z = LC_only_points['z'].min(), LC_only_points['z'].max()

core_vertices = new_core_mesh.vertices
core_faces = new_core_mesh.faces

def mesh_plane_intersection(vertices, faces, y_value):
    points = []
    for tri in faces:
        v0, v1, v2 = vertices[tri]
        y0, y1, y2 = v0[1], v1[1], v2[1]
        edges = [(v0, v1), (v1, v2), (v2, v0)]
        for v_start, v_end in edges:
            y_start, y_end = v_start[1], v_end[1]
            if (y_start - y_value) * (y_end - y_value) < 0:
                t = (y_value - y_start) / (y_end - y_start)
                intersect = v_start + t * (v_end - v_start)
                points.append(intersect)
    if points:
        points = np.array(points)
    else:
        points = np.empty((0, 3))
    return points

# Assign a color to each file
file_list = LC_only_points['file'].unique()
file_colors = {f: c for f, c in zip(file_list, colors[:len(file_list)])}
colors_list = plt.cm.tab10.colors
file_colors_tab10 = {f: colors_list[i % len(colors_list)] for i, f in enumerate(np.sort(file_list))}

if len(LC_only_points) > 36000:
    LC_only_points_sample = LC_only_points.sample(n=36000, random_state=42)
else:
    LC_only_points_sample = LC_only_points

fixed_xlim = (min_x, max_x+500)
fixed_ylim = (-max_z-200, -min_z+700)

svg_dir = '/root/capsule/results/slice_svgs'
os.makedirs(svg_dir, exist_ok=True)

n_bins = 50
fixed_bin_width = 10.0  # fixed bin width in PC2 projection units (adjust as needed)

# PASS 1: Find the global max histogram count and the baseline
#         length of the slice where that max occurs.
#         Also find global PC2 projection bounds for fixed baseline.
global_max_count = 0
baseline_length_at_max = 1.0
global_pc2_min = np.inf
global_pc2_max = -np.inf

for y_plane in y_planes:
    mask = (LC_only_points_sample['y'] > (y_plane - 50)) & (LC_only_points_sample['y'] < (y_plane + 50))
    lc_pts = LC_only_points_sample[mask].copy()

    if len(lc_pts) <= 10:
        continue

    xnz = lc_pts[['x', 'z']].values.copy()
    xnz[:, 1] = -xnz[:, 1]

    mean_xnz_all = xnz.mean(axis=0)
    std_xnz_all = xnz.std(axis=0)
    inlier_mask = np.all(np.abs(xnz - mean_xnz_all) <= 3 * std_xnz_all, axis=1)
    xnz_inlier = xnz[inlier_mask]
    file_vals_inlier = lc_pts['file'].values[inlier_mask]

    if len(xnz_inlier) <= 10:
        continue

    pca = PCA(n_components=2)
    pca.fit(xnz_inlier)
    pc2_vector = pca.components_[1]
    mean_xnz = pca.mean_

    projections_all = (xnz_inlier - mean_xnz) @ pc2_vector
    global_min_proj = projections_all.min()
    global_max_proj = projections_all.max()
    pc2_range = global_max_proj - global_min_proj

    # Track global PC2 bounds
    if global_min_proj < global_pc2_min:
        global_pc2_min = global_min_proj
    if global_max_proj > global_pc2_max:
        global_pc2_max = global_max_proj

    # Use fixed bin width
    bin_edges = np.arange(global_min_proj, global_max_proj + fixed_bin_width, fixed_bin_width)

    sorted_files = np.sort(np.unique(file_vals_inlier))
    for fname in sorted_files:
        proj_f = projections_all[file_vals_inlier == fname]
        if len(proj_f) < 5:
            continue
        counts, _ = np.histogram(proj_f, bins=bin_edges)
        counts_smooth = gaussian_filter1d(counts.astype(float), sigma=1.0)
        peak = counts_smooth.max()
        if peak > global_max_count:
            global_max_count = peak
            baseline_length_at_max = pc2_range

# Build the fixed global bin edges
baseline_trim = 500 
global_bin_edges = np.arange(global_pc2_min + baseline_trim, global_pc2_max - baseline_trim + fixed_bin_width, fixed_bin_width)
global_bin_centers = 0.5 * (global_bin_edges[:-1] + global_bin_edges[1:])
n_bins_global = len(global_bin_centers)

print(f"Global max smoothed count: {global_max_count:.1f}, "
      f"baseline length at that slice: {baseline_length_at_max:.1f}")
print(f"Global PC2 range: [{global_pc2_min:.1f}, {global_pc2_max:.1f}], "
      f"fixed bins: {n_bins_global}")

global_scale_factor = 0.9 * baseline_length_at_max / (global_max_count + 1e-6)

# PASS 2: Plot with the fixed global scale factor.
for i, y_plane in enumerate(y_planes):
    core_inter_pts = mesh_plane_intersection(core_vertices, core_faces, y_plane)

    mask = (LC_only_points_sample['y'] > (y_plane - 50)) & (LC_only_points_sample['y'] < (y_plane + 50))
    lc_pts = LC_only_points_sample[mask].copy()

    # Shuffle
    lc_pts = lc_pts.sample(frac=1, random_state=42).reset_index(drop=True)
    point_colors = [file_colors_tab10[f] for f in lc_pts['file']]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.scatter(lc_pts['x'], -lc_pts['z'], s=5, color=point_colors, alpha=0.3, linewidths=0)

    # Add mesh intersection line
    if len(core_inter_pts) > 0:
        ax.plot(core_inter_pts[:, 0], -core_inter_pts[:, 2], 'o', markersize=1, color='blue',
                label=f'Core Mesh @ y={y_plane}', alpha=0.5)

    # --- PC2 histogram overlay (with outlier exclusion) ---
    if len(lc_pts) > 10:
        xnz = lc_pts[['x', 'z']].values.copy()
        xnz[:, 1] = -xnz[:, 1]

        # Exclude outliers ±3 SD for PCA/histogram only
        mean_xnz_all = xnz.mean(axis=0)
        std_xnz_all = xnz.std(axis=0)
        inlier_mask = np.all(np.abs(xnz - mean_xnz_all) <= 3 * std_xnz_all, axis=1)
        xnz_inlier = xnz[inlier_mask]
        file_vals_inlier = lc_pts['file'].values[inlier_mask]

        if len(xnz_inlier) > 10:
            pca = PCA(n_components=2)
            pca.fit(xnz_inlier)
            pc1_vector = pca.components_[0]
            pc2_vector = pca.components_[1]
            mean_xnz = pca.mean_

            projections_all = (xnz_inlier - mean_xnz) @ pc2_vector

            # Use global fixed bin edges for consistent baseline width
            bin_edges = global_bin_edges
            bin_centers_1d = global_bin_centers
            n_bins_local = n_bins_global

            # Offset baseline along PC1
            projections_pc1 = (xnz_inlier - mean_xnz) @ pc1_vector
            if pc1_vector[1] < 0:
                baseline_offset = projections_pc1.min() - 50
                hist_sign = -1.0
            else:
                baseline_offset = projections_pc1.max() + 50
                hist_sign = 1.0

            def pc2_hist_to_xy(bin_centers, counts, scale):
                base = mean_xnz[None, :] + bin_centers[:, None] * pc2_vector[None, :]
                base = base + baseline_offset * pc1_vector[None, :]
                offsets = (hist_sign * counts)[:, None] * pc1_vector[None, :] * scale
                line = base + offsets
                return line[:, 0], line[:, 1]

            sorted_files = np.sort(np.unique(file_vals_inlier))

            # Draw baseline along PC2
            baseline_x, baseline_y = pc2_hist_to_xy(bin_centers_1d, np.zeros(n_bins_local), global_scale_factor)
            ax.plot(baseline_x, baseline_y, color='black', linewidth=1, linestyle='--', alpha=0.3)

            scalebar_count = 30
            tick_base_x, tick_base_y = pc2_hist_to_xy(bin_centers_1d[:1], np.array([0.0]), global_scale_factor)
            tick_top_x, tick_top_y = pc2_hist_to_xy(bin_centers_1d[:1], np.array([scalebar_count]), global_scale_factor)
            ax.plot([tick_base_x[0], tick_top_x[0]], [tick_base_y[0], tick_top_y[0]],
                    color='black', linewidth=2, solid_capstyle='butt', alpha=0.3)

            # Overlay histogram line per file
            for fname in sorted_files:
                fmask = file_vals_inlier == fname
                proj_f = projections_all[fmask]
                if len(proj_f) < 5:
                    continue
                counts, _ = np.histogram(proj_f, bins=bin_edges)
                counts_smooth = gaussian_filter1d(counts.astype(float), sigma=1.0)
                line_x, line_y = pc2_hist_to_xy(bin_centers_1d, counts_smooth, global_scale_factor)
                ax.plot(line_x, line_y, color=file_colors_tab10[fname], linewidth=2, alpha=0.85)

    # Add legend entries manually
    for f in file_list:
        if (lc_pts['file'] == f).any():
            ax.scatter([], [], s=15, color=file_colors_tab10[f],
                       label=f.replace('points_ccf_', '').replace('.npy', ''))

    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax.set_title(f'LC_only_points at y={y_plane} (±50), colored by file')
    if i == 0:
        ax.legend(markerscale=3, loc='upper right', fontsize=8)
    ax.set_aspect('equal')
    ax.set_xlim(fixed_xlim)
    ax.set_ylim(fixed_ylim)

    # Save as SVG
    svg_path = os.path.join(svg_dir, f'LC_slice_y{y_plane}.svg')
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    print(f'Saved {svg_path}')

# SELF REGISTRATION

In [ ]:
groups = LC_only_points.groupby(['file', 'reflected']).size().reset_index(name='count')
print("Unique (file, reflected) groups:", len(groups))
print(groups)

# Build coarse voxel occupancy per group and pick the group whose voxel occupancy
# is closest to the average occupancy across all groups (L2 distance).
coords_all = LC_only_points[['x','y','z']].values.astype(np.float32)
mins = coords_all.min(axis=0)
maxs = coords_all.max(axis=0)
print("Global bounds:", mins, maxs)

# voxel grid resolution
nx = ny = nz = 32

edges = [
    np.linspace(mins[0], maxs[0], nx+1),
    np.linspace(mins[1], maxs[1], ny+1),
    np.linspace(mins[2], maxs[2], nz+1),
]

group_keys = list(LC_only_points.groupby(['file','reflected']).groups.keys())
occ_list = []
labels = []

for key in group_keys:
    df_g = LC_only_points[(LC_only_points['file']==key[0]) & (LC_only_points['reflected']==key[1])]
    pts = df_g[['x','y','z']].values
    if len(pts) == 0:
        occ = np.zeros((nx,ny,nz), dtype=np.float32)
    else:
        h, _ = np.histogramdd(pts, bins=edges)
        occ = (h>0).astype(np.float32)  # occupancy
        occ = occ / (occ.sum() + 1e-12)  # normalize
    occ_list.append(occ.ravel())
    labels.append(key)

occ_stack = np.vstack(occ_list)  # (n_groups, voxels)
global_mean_occ = occ_stack.mean(axis=0)

# compute L2 distance to the mean occupancy
dists = np.linalg.norm(occ_stack - global_mean_occ[None, :], axis=1)
best_idx = int(np.argmin(dists))
ref_key = labels[best_idx]
print(f"Selected reference group (closest to mean occupancy): {ref_key} (index {best_idx}), group size = {groups.loc[(groups['file']==ref_key[0]) & (groups['reflected']==ref_key[1]), 'count'].values[0]}")
print("L2 distances (sorted):")
for i in np.argsort(dists)[:12]:
    print(f"  {labels[i]} -> {dists[i]:.6f}")

ref_df = LC_only_points[(LC_only_points['file']==ref_key[0]) & (LC_only_points['reflected']==ref_key[1])]
ref_pts = ref_df[['x','y','z']].values.astype(np.float32)
print("Reference points available:", ref_pts.shape)

In [ ]:
def compute_emd(X, Y, numItermax=1000000):

    if len(X) == 0 or len(Y) == 0:
        return 1e10
    
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    
    # Normalize to [0, 1] range
    all_pts = np.vstack([X, Y])
    shift = all_pts.min(axis=0)
    scale = (all_pts.max(axis=0) - shift)
    scale[scale == 0] = 1.0
    
    X_norm = (X - shift) / scale
    Y_norm = (Y - shift) / scale
    
    # Compute pairwise distances
    M = cdist(X_norm, Y_norm, metric='euclidean')
    M = np.ascontiguousarray(M, dtype=np.float64)
    
    # Uniform weights
    a = np.ones(len(X_norm), dtype=np.float64) / len(X_norm)
    b = np.ones(len(Y_norm), dtype=np.float64) / len(Y_norm)
    
    # Compute EMD
    emd_val = ot.emd2(a, b, M, numItermax=numItermax)
    
    # Scale back
    mean_scale = np.mean(scale)
    return emd_val * mean_scale

def apply_rigid_transform(points, R, t):

    return (R @ points.T).T + t

def rotation_matrix_from_params(params):

    rx, ry, rz = params
    
    # Rotation matrices
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx), np.cos(rx)]
    ])
    
    Ry = np.array([
        [np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])
    
    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz), np.cos(rz), 0],
        [0, 0, 1]
    ])
    
    return Rz @ Ry @ Rx

def register_to_reference(moving_points, ref_points, init_params=None, verbose=False):
    
    if init_params is None:
        init_params = np.zeros(6)
    
    # Convert to float64
    moving_points = np.asarray(moving_points, dtype=np.float64)
    ref_points = np.asarray(ref_points, dtype=np.float64)
    
    eval_count = [0]
    
    def objective(params):
        eval_count[0] += 1
        rx, ry, rz = params[:3]
        tx, ty, tz = params[3:]
        
        R = rotation_matrix_from_params([rx, ry, rz])
        t = np.array([tx, ty, tz])
        
        transformed = apply_rigid_transform(moving_points, R, t)
        emd = compute_emd(transformed, ref_points, numItermax=1000000)
        
        if verbose and eval_count[0] % 50 == 0:
            print(f"  Eval {eval_count[0]}: EMD={emd:.4f}")
        
        return emd
    
    bounds = [(-np.pi, np.pi)] * 3 + [(-5000, 5000)] * 3
    
    result = minimize(
        objective,
        init_params,
        method='L-BFGS-B',
        bounds=bounds,
        options={
            'maxiter': 3000,
            'ftol': 1e-7,
            'gtol': 1e-6,
            'maxcor': 30,
            'maxfun': 5000,
        }
    )
    
    R_final = rotation_matrix_from_params(result.x[:3])
    t_final = result.x[3:]
    emd_final = result.fun
    
    return {
        'params': result.x,
        'R': R_final,
        't': t_final,
        'emd': emd_final,
        'success': result.success,
        'n_evals': eval_count[0],
        'n_iters': result.nit
    }

In [ ]:
ref_pts = ref_pts.astype(np.float32)
print(f"Reference group {ref_key}: {ref_pts.shape[0]} points\n")

results = {}
group_keys = sorted(list(LC_only_points.groupby(['file', 'reflected']).groups.keys()))

print(f"Registering {len(group_keys)} groups to reference {ref_key}...\n")

failed_registrations = []

for key in tqdm(group_keys):
    if key == ref_key:
        results[key] = {
            'R': np.eye(3),
            't': np.zeros(3),
            'emd': 0.0,
            'success': True,
            'n_evals': 0,
            'n_iters': 0
        }
        print(f"  {key}: REFERENCE (identity)")
        continue
    
    fname, reflected = key
    df_g = LC_only_points[(LC_only_points['file'] == fname) & 
                          (LC_only_points['reflected'] == reflected)]
    moving_pts = df_g[['x', 'y', 'z']].values.astype(np.float32)
    
    if len(moving_pts) < 10:
        print(f"  {key}: SKIPPED ({len(moving_pts)} points)")
        continue
    
    # Try registration with verbose output
    reg_result = register_to_reference(moving_pts, ref_pts, verbose=True)
    
    results[key] = reg_result
    
    status = "✓" if reg_result['success'] else "✗"
    print(f"  {key}: EMD={reg_result['emd']:.2f}, iters={reg_result['n_iters']}, "
          f"evals={reg_result['n_evals']}, success={status}\n")
    
    if not reg_result['success']:
        failed_registrations.append((key, reg_result['emd']))

print(f"\nCompleted registration of {len(results)} groups")
print(f"Failed registrations: {len(failed_registrations)}")

if failed_registrations:
    print("\nFailed registrations (highest EMD):")
    for key, emd in sorted(failed_registrations, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {key}: EMD={emd:.2f}")

# Summary statistics
emd_values = [v['emd'] for v in results.values() if v['emd'] > 0]
print(f"\nEMD Statistics (excluding reference):")
print(f"  Mean: {np.mean(emd_values):.2f}")
print(f"  Std:  {np.std(emd_values):.2f}")
print(f"  Min:  {np.min(emd_values):.2f}")
print(f"  Max:  {np.max(emd_values):.2f}")
print(f"  Median: {np.median(emd_values):.2f}")

In [ ]:
# Add registered coordinates columns to LC_only_points
LC_only_points['reg_x'] = LC_only_points['x'].copy()
LC_only_points['reg_y'] = LC_only_points['y'].copy()
LC_only_points['reg_z'] = LC_only_points['z'].copy()

# Apply registration transforms for each group
for key in results.keys():
    fname, reflected = key
    mask = (LC_only_points['file'] == fname) & (LC_only_points['reflected'] == reflected)
    
    if mask.sum() == 0:
        continue
    
    # Get the points for this group
    pts = LC_only_points.loc[mask, ['x', 'y', 'z']].values.astype(np.float64)
    
    # Get transformation
    R = results[key]['R']
    t = results[key]['t']
    
    # Apply rigid transformation
    transformed = (R @ pts.T).T + t
    
    # Cast to float32 before assigning to avoid dtype mismatch warning
    transformed = transformed.astype(np.float32)
    
    # Update the registered coordinate columns
    LC_only_points.loc[mask, 'reg_x'] = transformed[:, 0]
    LC_only_points.loc[mask, 'reg_y'] = transformed[:, 1]
    LC_only_points.loc[mask, 'reg_z'] = transformed[:, 2]

print(f"LC_only_points shape: {LC_only_points.shape}")
print(LC_only_points[['file', 'x', 'y', 'z', 'reg_x', 'reg_y', 'reg_z']].head())

In [ ]:
# Calculate registration error (Euclidean distance between original and registered coordinates)
orig = LC_only_points[['x', 'y', 'z']].values.astype(np.float64)
reg = LC_only_points[['reg_x', 'reg_y', 'reg_z']].values.astype(np.float64)

LC_only_points['reg_error'] = np.linalg.norm(reg - orig, axis=1)

In [ ]:
df_pos = LC_only_points[(LC_only_points['reg_error'] > 0) & (LC_only_points['in_new_core_mesh'] == 1)]
grp_means = df_pos.groupby(['file','reflected'])['reg_error'].mean()
print("Per-group means:")
print(grp_means)
print(f"\nMean of the means: {grp_means.mean():.4f}")

# assume LC_only_points already in namespace with columns x,y,z, reg_x,reg_y,reg_z
orig = LC_only_points[['x', 'y', 'z']].values.astype(np.float64)
reg = LC_only_points[['reg_x', 'reg_y', 'reg_z']].values.astype(np.float64)

dists = np.linalg.norm(reg - orig, axis=1)
LC_only_points['reg_error'] = dists  # store per-point errors

median_distance = float(np.median(dists))
mean_distance = float(np.mean(dists))

print(f"Median distance (reg vs orig): {median_distance:.4f}")
print(f"Mean distance (reg vs orig):   {mean_distance:.4f}")

# optional: per-file median summary
per_file_median = LC_only_points.groupby('file')['reg_error'].median().sort_values()
print("\nPer-file median registration error:")
print(per_file_median)

# Recreate Core Mesh with self-registered points

In [ ]:
# local density calculation
coords = LC_only_points[['reg_x', 'reg_y', 'reg_z']].values
nbrs = NearestNeighbors(n_neighbors=101, algorithm='auto').fit(coords)
distances, _ = nbrs.kneighbors(coords)
# Exclude distance to self
LC_only_points['kNN'] = distances[:, 1:101].mean(axis=1)

# Compute percentiles for LC_only_points['kNN'] with respect to all kNN values
percentiles = rankdata(LC_only_points['kNN'], method='average') / len(LC_only_points) * 100
LC_only_points['kNN_percentile'] = percentiles

In [ ]:
# Generate mask for selected points
mask = (LC_only_points['kNN_percentile'] > 10) & (LC_only_points['kNN_percentile'] < slider.value)
selected_points = LC_only_points.loc[mask, ['reg_x', 'reg_y', 'reg_z']].values
print(f"Selected {selected_points.shape[0]} points with 10 < kNN_percentile < {slider.value}")

# Select internal points at core
internal_mask = LC_only_points['kNN_percentile'] < 10
internal_points = LC_only_points.loc[internal_mask, ['reg_x', 'reg_y', 'reg_z']].values
print(f"Selected {internal_points.shape[0]} internal points with kNN < 10")

In [ ]:
# CORE MESH ESTIMATOR
shell_points = selected_points.astype(np.float32)

interior_points = internal_points.astype(np.float32)

normals_normalized = estimate_normals(shell_points, k=80)
consistently_oriented_normals = orient_complex_shape_normals(
    shell_points,
    normals_normalized.copy(),
    interior_points=interior_points
)
consistently_oriented_normals = np.asfortranarray(consistently_oriented_normals, dtype=np.float32)

vertices, faces = pcu.pointcloud_surfel_geometry(shell_points, consistently_oriented_normals, 30)

vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices, faces, 10000)
mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

while not mesh.is_watertight:
    print("Mesh is not watertight, retrying...")
    vertices_wt, faces_wt = pcu.make_mesh_watertight(vertices_wt, faces_wt, 10000)
    mesh = trimesh.Trimesh(vertices=vertices_wt, faces=faces_wt)

# Convert trimesh to Open3D
mesh_o3d = o3d.geometry.TriangleMesh(
    o3d.utility.Vector3dVector(mesh.vertices),
    o3d.utility.Vector3iVector(mesh.faces)
)

# Optional: compute vertex normals if needed
mesh_o3d.compute_vertex_normals()

# number_of_iterations: how aggressive
# lambda: smoothing strength (0 < λ < 1)
mesh_o3d = mesh_o3d.filter_smooth_simple(number_of_iterations=5)

# Convert back to trimesh
mesh_smoothed = trimesh.Trimesh(
    np.asarray(mesh_o3d.vertices),
    np.asarray(mesh_o3d.triangles)
)
self_reg_core_mesh = mesh_smoothed

In [ ]:
# Compute curvature and voxelize
test_mesh = self_reg_core_mesh
L = trimesh.smoothing.laplacian_calculation(test_mesh, equal_weight=False)
signed_curvature = np.sign(np.einsum('ij,ij->i', L.dot(test_mesh.vertices), test_mesh.vertex_normals)) * np.linalg.norm(L.dot(test_mesh.vertices), axis=1)

# Distance transform from surface
pitch = 3
voxelized = test_mesh.voxelized(pitch=pitch).fill()
distance = np.zeros_like(voxelized.matrix, dtype=np.uint8)
frontier = ndimage.binary_erosion(voxelized.matrix) ^ voxelized.matrix

for d in tqdm(range(1, 6), desc="Computing distance transform"):
    distance[frontier] = d
    frontier = ndimage.binary_dilation(frontier) & voxelized.matrix & (distance == 0)

distance[distance == 0] = 6

# Map vertices to distance values and filter
tree = cKDTree(voxelized.points)
_, nearest_idx = tree.query(test_mesh.vertices, k=1)
vertex_distance = distance[voxelized.matrix][nearest_idx]

keep_mask = vertex_distance <= 2
old_to_new = -np.ones(len(test_mesh.vertices), dtype=int)
old_to_new[keep_mask] = np.arange(np.sum(keep_mask))

face_mask = keep_mask[test_mesh.faces].all(axis=1)
filtered_faces = old_to_new[test_mesh.faces[face_mask]]

# Repair and seal
sealed_mesh = trimesh.Trimesh(vertices=test_mesh.vertices[keep_mask], faces=filtered_faces)
trimesh.repair.fix_winding(sealed_mesh)
seal_holes(sealed_mesh)
trimesh.repair.fill_holes(sealed_mesh)
trimesh.repair.fix_normals(sealed_mesh, multibody=True)
sealed_mesh.is_watertight

In [ ]:
output_dir = '/root/capsule/results/percentile_meshes'
output_path = os.path.join(output_dir, f'self_registered_core_mesh.obj')
sealed_mesh.export(output_path)

In [ ]:
# paths
p_new = '/root/capsule/results/percentile_meshes/new_core_mesh.obj'
p_self = '/root/capsule/results/percentile_meshes/self_registered_core_mesh.obj'

for p in (p_new, p_self):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Mesh not found: {p}")

# load meshes without heavy processing
mesh_new = trimesh.load(p_new, process=False)
mesh_self = trimesh.load(p_self, process=False)

fig = go.Figure()

# add mesh traces
fig.add_trace(go.Mesh3d(
    x=mesh_new.vertices[:, 0],
    y=mesh_new.vertices[:, 1],
    z=mesh_new.vertices[:, 2],
    i=mesh_new.faces[:, 0],
    j=mesh_new.faces[:, 1],
    k=mesh_new.faces[:, 2],
    color='royalblue',
    opacity=0.45,
    name='new_core_mesh'
))

fig.add_trace(go.Mesh3d(
    x=mesh_self.vertices[:, 0],
    y=mesh_self.vertices[:, 1],
    z=mesh_self.vertices[:, 2],
    i=mesh_self.faces[:, 0],
    j=mesh_self.faces[:, 1],
    k=mesh_self.faces[:, 2],
    color='crimson',
    opacity=0.5,
    name='self_registered_core_mesh'
))

fig.update_layout(
    title='Overlay: new_core_mesh (blue) vs self_registered_core_mesh (red)',
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='data',
        camera=dict(projection=dict(type='orthographic'))
    ),
    showlegend=True
)

fig.show()

# Load both meshes
mesh_new = trimesh.load(p_new, process=False)
mesh_self = trimesh.load(p_self, process=False)

# Compute volumes
vol_new = mesh_new.volume
vol_self = mesh_self.volume

print(f"Volume of new_core_mesh: {vol_new:.2f} cubic micrometers")
print(f"Volume of self_registered_core_mesh: {vol_self:.2f} cubic micrometers")
print(f"Difference: {abs(vol_new - vol_self):.2f} cubic micrometers")
print(f"Percent difference: {100 * abs(vol_new - vol_self) / vol_new:.2f}%")